# 05 — Selección de longitudes de onda y simulación de sensor simplificado

## Proyecto
**Kaki_DeepHS_StorageDays**

## Objetivo

Determinar si el desempeño de `VIS_COR` puede conservarse utilizando un número pequeño de longitudes de onda, en lugar de las 249 bandas originales.

Se compararán paneles virtuales de:

\[
k \in \{3,5,8,10,15,20\}
\]

bandas contra el espectro completo.

## Principios metodológicos

### 1. La selección de bandas ocurre dentro de training

Para la evaluación estándar, las bandas de cada outer fold se seleccionan **solo con los 32 frutos de desarrollo**.

Durante la selección de hiperparámetros interna, las bandas se vuelven a seleccionar utilizando únicamente los frutos de training de cada inner fold.

### 2. La selección busca señal longitudinal, no diferencias estáticas entre frutos

Para cada longitud de onda se calcula la correlación temporal **dentro de cada fruto** y luego se resume con el mismo peso por fruto.

Esto reduce la posibilidad de seleccionar bandas únicamente por diferencias estables entre frutos.

### 3. Restricción de separación espectral

Las bandas seleccionadas deben estar separadas al menos:

\[
10\ {\rm nm}
\]

para evitar elegir múltiples bandas casi redundantes y acercarnos a un posible diseño multiespectral real.

### 4. Preprocesamiento compatible con sensor multibanda

Para paneles dispersos se comparan:

- RAW
- SNV

No se utiliza SG1 en el sensor sparse porque una derivada Savitzky–Golay requiere vecindad espectral densa y no corresponde directamente a un sensor de pocas bandas discretas.

### 5. Session-excluded

El panel de bandas de cada outer fold queda **fijo** después de ser seleccionado con sus 32 frutos de desarrollo.

Luego se hace un stress test donde, para predecir el día \(d\), el **modelo** se ajusta excluyendo todas las filas de día/sesión \(d\).

> Importante: este stress test no es deconfounding causal completo porque la etapa de diseño del panel sparse sí observó todos los días de los 32 frutos de desarrollo. Por eso se interpreta como robustez del sensor seleccionado, no como validación independiente de sesión.

### 6. Null correcto para session-excluded

Se calcula un `SessionExcluded_DummyMean` que también elimina del training el día objetivo. Así el null sufre exactamente la misma ausencia de etiqueta temporal que el modelo.

## Resultado esperado

Responder:

> ¿Cuántas longitudes de onda son necesarias para conservar la mayor parte del desempeño del VIS_COR completo?


### Public reproducibility version

This notebook is the cleaned repository version used to reproduce the manuscript analyses. The scientific workflow, frozen fruit-disjoint partitions, model-selection logic, seeds, and evaluation metrics are preserved. Google Drive mounting and chat-specific ZIP export steps were removed. Paths are relative to the repository root. Raw DeepHS Fruit data are not redistributed; see `../docs/DATASET.md`.


In [ ]:
from pathlib import Path
import os

def find_repository_root():
    """Locate the cloned repository or use PERSIMMON_HSI_ROOT if defined."""
    env_root = os.environ.get("PERSIMMON_HSI_ROOT")
    if env_root:
        return Path(env_root).expanduser().resolve()

    cwd = Path.cwd().resolve()
    candidates = [cwd, *cwd.parents]
    for candidate in candidates:
        if (candidate / "notebooks").is_dir() and (candidate / "README.md").is_file():
            return candidate

    if cwd.name == "notebooks":
        return cwd.parent

    return cwd

PROJECT_ROOT = find_repository_root()
print("Repository root:", PROJECT_ROOT)


## 1. Librerías, rutas y configuración


In [ ]:
from pathlib import Path
from datetime import datetime
import hashlib
import itertools
import json
import math
import platform
import shutil
import sys
import time
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.stats import spearmanr

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
)

from tqdm.auto import tqdm

warnings.filterwarnings("ignore", category=RuntimeWarning)

PROJECT_NAME = "Kaki_DeepHS_StorageDays"
NOTEBOOK_ID = "05_kaki_wavelength_selection_sensor_simplification"
RUN_SIGNATURE = "viscor_sparse_sensor_v1"

ROOT = PROJECT_ROOT

INTERIM = ROOT / "data/interim"
SPLITS_DIR = ROOT / "data/splits"

BASE02 = ROOT / "results/02_kaki_fruit_disjoint_splits_baselines"
BASE03 = ROOT / "results/03_kaki_robustness_session_bias"

RESULTS = ROOT / "results/05_kaki_wavelength_selection_sensor_simplification"
FIGURES = ROOT / "figures/05_kaki_wavelength_selection_sensor_simplification"
TABLES = ROOT / "tables/05_kaki_wavelength_selection_sensor_simplification"
EXPORT_ROOT = ROOT / "exports/05_kaki_wavelength_selection_sensor_simplification"

for p in [
    RESULTS,
    FIGURES,
    TABLES,
    EXPORT_ROOT,
]:
    p.mkdir(parents=True, exist_ok=True)

CAMERA = "VIS_COR"

K_VALUES = [
    3,
    5,
    8,
    10,
    15,
    20,
]

MIN_BAND_SPACING_NM = 10.0

PREPROCESSING = [
    "raw",
    "snv",
]

RIDGE_ALPHAS = [
    0.01,
    0.1,
    1.0,
    10.0,
    100.0,
    1000.0,
]

OUTER_FOLDS = 5
INNER_FOLDS = 4

RANDOM_SEED = 20260818
BOOTSTRAP_REPS = 5000

print("ROOT:", ROOT)
print("Sensor:", CAMERA)
print("k:", K_VALUES)
print("Separación mínima:", MIN_BAND_SPACING_NM, "nm")


## 2. Cargar VIS_COR aprobado, splits congelados y referencias de notebooks 02–03


In [ ]:
SPECTRA_PATH = (
    INTERIM /
    "kaki_fruit_day_spectra_VIS_COR.csv"
)

CUBE_PATH = (
    INTERIM /
    "kaki_cube_spectra_VIS_COR.csv"
)

OUTER_SPLIT_PATH = (
    SPLITS_DIR /
    "kaki_outer5_fruit_disjoint_v1.csv"
)

INNER_SPLIT_PATH = (
    SPLITS_DIR /
    "kaki_inner4_within_outer5_fruit_disjoint_v1.csv"
)

BASELINE_OOF_PATH = (
    BASE02 /
    "kaki_oof_predictions.csv"
)

SESSION_OOF_PATH = (
    BASE03 /
    "kaki_session_excluded_oof_predictions.csv"
)

for p in [
    SPECTRA_PATH,
    CUBE_PATH,
    OUTER_SPLIT_PATH,
    INNER_SPLIT_PATH,
    BASELINE_OOF_PATH,
    SESSION_OOF_PATH,
]:
    if not p.exists():
        raise FileNotFoundError(
            f"Falta archivo requerido: {p}"
        )

cube = pd.read_csv(
    CUBE_PATH
)

assert len(cube) == 398
assert float(
    cube["mask_coverage"].median()
) < 0.87
assert (
    cube["mask_qc_flag"] == "ok"
).all()
assert cube[
    "extraction_error"
].isna().all()

data = pd.read_csv(
    SPECTRA_PATH
)

data["fruit_id"] = pd.to_numeric(
    data["fruit_id"],
    errors="raise"
).astype(int)

data["storage_days_model"] = pd.to_numeric(
    data["storage_days_model"],
    errors="raise"
).astype(int)

wl_cols = [
    c
    for c in data.columns
    if c.startswith("wl_")
]

wavelengths = np.array(
    [
        float(
            c.replace(
                "wl_",
                ""
            )
        )
        for c in wl_cols
    ],
    dtype=float
)

assert len(data) == 199
assert data["fruit_id"].nunique() == 40
assert len(wl_cols) == 249
assert not data[wl_cols].isna().any().any()

data = data.sort_values(
    [
        "fruit_id",
        "storage_days_model"
    ]
).reset_index(drop=True)

outer = pd.read_csv(
    OUTER_SPLIT_PATH
)

inner = pd.read_csv(
    INNER_SPLIT_PATH
)

baseline_oof = pd.read_csv(
    BASELINE_OOF_PATH
)

session_oof_full = pd.read_csv(
    SESSION_OOF_PATH
)

for df in [
    outer,
    inner,
]:
    df["fruit_id"] = pd.to_numeric(
        df["fruit_id"],
        errors="raise"
    ).astype(int)

outer["outer_fold"] = pd.to_numeric(
    outer["outer_fold"],
    errors="raise"
).astype(int)

inner["outer_fold"] = pd.to_numeric(
    inner["outer_fold"],
    errors="raise"
).astype(int)

inner["inner_fold"] = pd.to_numeric(
    inner["inner_fold"],
    errors="raise"
).astype(int)

print(
    "VIS_COR:",
    len(data),
    "fruit-days,",
    data["fruit_id"].nunique(),
    "frutos,",
    len(wavelengths),
    "bandas"
)

print(
    f"Rango: {wavelengths.min():.3f}–"
    f"{wavelengths.max():.3f} nm"
)

print("✅ Inputs cargados.")


## 3. Auditoría de splits


In [ ]:
audit_rows = []

all_fruits = set(
    outer[
        "fruit_id"
    ].astype(int)
)

for fold in range(
    OUTER_FOLDS
):

    test_fruits = set(
        outer.loc[
            outer[
                "outer_fold"
            ]
            ==
            fold,
            "fruit_id"
        ].astype(int)
    )

    train_fruits = (
        all_fruits
        -
        test_fruits
    )

    inner_fruits = set(
        inner.loc[
            inner[
                "outer_fold"
            ]
            ==
            fold,
            "fruit_id"
        ].astype(int)
    )

    audit_rows.append({
        "outer_fold":
            fold,

        "n_train_fruits":
            len(
                train_fruits
            ),

        "n_test_fruits":
            len(
                test_fruits
            ),

        "train_test_overlap":
            len(
                train_fruits
                &
                test_fruits
            ),

        "inner_test_overlap":
            len(
                inner_fruits
                &
                test_fruits
            ),

        "inner_matches_train":
            inner_fruits
            ==
            train_fruits,
    })

split_audit = pd.DataFrame(
    audit_rows
)

display(
    split_audit
)

assert (
    split_audit[
        "train_test_overlap"
    ]
    ==
    0
).all()

assert (
    split_audit[
        "inner_test_overlap"
    ]
    ==
    0
).all()

assert split_audit[
    "inner_matches_train"
].all()

split_audit.to_csv(
    TABLES /
    "01_split_audit.csv",
    index=False
)

print("✅ Sin fuga de identidad.")


# 4. Score longitudinal para selección de bandas

Para cada fruto de training y para cada banda se calcula la correlación de Pearson entre reflectancia y `storage_days` **dentro del mismo fruto**.

Para la banda \(\lambda\):

\[
r_{i,\lambda}
=
cor(
x_{i,\lambda}(t),
t
)
\]

Luego:

\[
score_\lambda
=
\left|
\frac{1}{N}
\sum_i r_{i,\lambda}
\right|
\times
consistency_\lambda
\]

donde `consistency` es la fracción de frutos que comparten la dirección temporal dominante.

Así cada fruto contribuye con el mismo peso al score de selección.


In [ ]:
def within_fruit_temporal_band_scores(
    df,
    spectral_columns
):

    per_fruit_r = []

    used_fruits = []

    for fruit_id, g in df.groupby(
        "fruit_id"
    ):

        g = g.sort_values(
            "storage_days_model"
        )

        if (
            g[
                "storage_days_model"
            ].nunique()
            <
            3
        ):
            continue

        y = g[
            "storage_days_model"
        ].to_numpy(
            dtype=float
        )

        X = g[
            spectral_columns
        ].to_numpy(
            dtype=float
        )

        yc = (
            y
            -
            y.mean()
        )

        Xc = (
            X
            -
            X.mean(
                axis=0,
                keepdims=True
            )
        )

        num = np.sum(
            Xc
            *
            yc[
                :,
                None
            ],
            axis=0
        )

        den = np.sqrt(
            np.sum(
                Xc ** 2,
                axis=0
            )
            *
            np.sum(
                yc ** 2
            )
        )

        r = np.divide(
            num,
            den,
            out=np.full(
                X.shape[
                    1
                ],
                np.nan,
                dtype=float
            ),
            where=den > 0
        )

        per_fruit_r.append(
            r
        )

        used_fruits.append(
            int(
                fruit_id
            )
        )

    if not per_fruit_r:
        raise RuntimeError(
            "No hay frutos con trayectoria suficiente."
        )

    R = np.vstack(
        per_fruit_r
    )

    mean_r = np.nanmean(
        R,
        axis=0
    )

    median_r = np.nanmedian(
        R,
        axis=0
    )

    positive_fraction = np.nanmean(
        R > 0,
        axis=0
    )

    negative_fraction = np.nanmean(
        R < 0,
        axis=0
    )

    sign_consistency = np.maximum(
        positive_fraction,
        negative_fraction
    )

    score = (
        np.abs(
            mean_r
        )
        *
        sign_consistency
    )

    out = pd.DataFrame({
        "column":
            spectral_columns,

        "wavelength_nm":
            [
                float(
                    c.replace(
                        "wl_",
                        ""
                    )
                )
                for c in spectral_columns
            ],

        "mean_within_fruit_r":
            mean_r,

        "median_within_fruit_r":
            median_r,

        "sign_consistency":
            sign_consistency,

        "selection_score":
            score,

        "n_training_fruits_used":
            len(
                used_fruits
            ),
    })

    return (
        out.sort_values(
            "selection_score",
            ascending=False
        )
        .reset_index(
            drop=True
        )
    )


def select_spaced_bands(
    score_table,
    k,
    min_spacing_nm=10.0
):

    chosen = []

    for _, row in score_table.iterrows():

        wl = float(
            row[
                "wavelength_nm"
            ]
        )

        if all(
            abs(
                wl
                -
                selected[
                    "wavelength_nm"
                ]
            )
            >=
            min_spacing_nm
            for selected in chosen
        ):

            chosen.append(
                row.to_dict()
            )

        if len(
            chosen
        ) == k:
            break

    if len(
        chosen
    ) < k:

        raise RuntimeError(
            f"No se pudieron seleccionar {k} bandas "
            f"con spacing={min_spacing_nm} nm."
        )

    selected = pd.DataFrame(
        chosen
    )

    selected[
        "selection_rank"
    ] = np.arange(
        1,
        len(
            selected
        )
        +
        1
    )

    return selected


demo_scores = (
    within_fruit_temporal_band_scores(
        data,
        wl_cols
    )
)

demo_selected = select_spaced_bands(
    demo_scores,
    k=10,
    min_spacing_nm=MIN_BAND_SPACING_NM
)

display(
    demo_selected[
        [
            "selection_rank",
            "wavelength_nm",
            "selection_score",
            "mean_within_fruit_r",
            "sign_consistency",
        ]
    ]
)

print("✅ Selector longitudinal funcionando.")


## 5. Visualizar el score longitudinal usando todos los frutos

**Solo descriptivo.** Esta figura no se utiliza para estimar desempeño OOF.


In [ ]:
fig, ax = plt.subplots(
    figsize=(10, 4.8)
)

ax.plot(
    demo_scores[
        "wavelength_nm"
    ],
    demo_scores[
        "selection_score"
    ]
)

for _, row in demo_selected.iterrows():

    ax.axvline(
        row[
            "wavelength_nm"
        ],
        alpha=0.3
    )

ax.set_xlabel(
    "Longitud de onda (nm)"
)

ax.set_ylabel(
    "Score temporal longitudinal"
)

ax.set_title(
    "VIS_COR — score longitudinal de selección\n"
    "(todos los frutos; descriptivo)"
)

ax.grid(
    alpha=0.15
)

fig.tight_layout()

fig.savefig(
    FIGURES /
    "01_longitudinal_band_selection_score_all_fruits.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()


# 6. Preprocesamiento sparse y métricas


In [ ]:
def preprocess_sparse(
    X,
    method
):

    X = np.asarray(
        X,
        dtype=float
    )

    if method == "raw":
        return X.copy()

    if method == "snv":

        mu = np.mean(
            X,
            axis=1,
            keepdims=True
        )

        sd = np.std(
            X,
            axis=1,
            keepdims=True
        )

        sd[
            (~np.isfinite(sd))
            |
            (sd == 0)
        ] = 1.0

        return (
            X - mu
        ) / sd

    raise ValueError(
        method
    )


def fit_predict_ridge(
    X_train,
    y_train,
    X_test,
    alpha
):

    scaler = StandardScaler()

    X_train_s = scaler.fit_transform(
        X_train
    )

    X_test_s = scaler.transform(
        X_test
    )

    model = Ridge(
        alpha=float(
            alpha
        )
    )

    model.fit(
        X_train_s,
        y_train
    )

    pred = model.predict(
        X_test_s
    )

    return np.asarray(
        pred,
        dtype=float
    )


def fruit_balanced_mae(
    prediction_df
):

    tmp = prediction_df.copy()

    tmp[
        "abs_error"
    ] = np.abs(
        tmp[
            "y_true"
        ]
        -
        tmp[
            "y_pred"
        ]
    )

    return float(
        tmp.groupby(
            "fruit_id"
        )[
            "abs_error"
        ]
        .mean()
        .mean()
    )


def day_balanced_mae(
    prediction_df
):

    tmp = prediction_df.copy()

    tmp[
        "abs_error"
    ] = np.abs(
        tmp[
            "y_true"
        ]
        -
        tmp[
            "y_pred"
        ]
    )

    return float(
        tmp.groupby(
            "y_true"
        )[
            "abs_error"
        ]
        .mean()
        .mean()
    )


def metric_bundle(
    prediction_df
):

    y_true = prediction_df[
        "y_true"
    ].to_numpy(
        dtype=float
    )

    y_pred = prediction_df[
        "y_pred"
    ].to_numpy(
        dtype=float
    )

    residual = (
        y_pred
        -
        y_true
    )

    rho, _ = spearmanr(
        y_true,
        y_pred
    )

    return {
        "n_observations":
            int(
                len(
                    prediction_df
                )
            ),

        "n_fruits":
            int(
                prediction_df[
                    "fruit_id"
                ].nunique()
            ),

        "mae":
            float(
                mean_absolute_error(
                    y_true,
                    y_pred
                )
            ),

        "rmse":
            float(
                np.sqrt(
                    mean_squared_error(
                        y_true,
                        y_pred
                    )
                )
            ),

        "r2":
            float(
                r2_score(
                    y_true,
                    y_pred
                )
            ),

        "spearman_rho":
            float(
                rho
            ),

        "within_1_day_accuracy":
            float(
                np.mean(
                    np.abs(
                        residual
                    )
                    <=
                    1.0
                )
            ),

        "fruit_balanced_mae":
            fruit_balanced_mae(
                prediction_df
            ),

        "day_balanced_mae":
            day_balanced_mae(
                prediction_df
            ),

        "bias":
            float(
                np.mean(
                    residual
                )
            ),
    }


# 7. Nested CV estándar para un panel sparse

Para cada inner fold:

1. seleccionar bandas solo con inner-training;
2. aplicar esas bandas a inner-validation;
3. ajustar `RAW/SNV + Ridge(alpha)`.

El test externo nunca interviene en selección de bandas ni hiperparámetros.


In [ ]:
def inner_select_standard_sparse(
    development_df,
    inner_assignment,
    k
):

    rows = []

    best = None

    for preprocessing, alpha in itertools.product(
        PREPROCESSING,
        RIDGE_ALPHAS
    ):

        pred_parts = []

        for inner_fold in range(
            INNER_FOLDS
        ):

            val_fruits = set(
                inner_assignment.loc[
                    inner_assignment[
                        "inner_fold"
                    ]
                    ==
                    inner_fold,
                    "fruit_id"
                ].astype(int)
            )

            train_df = development_df[
                ~development_df[
                    "fruit_id"
                ].isin(
                    val_fruits
                )
            ].copy()

            val_df = development_df[
                development_df[
                    "fruit_id"
                ].isin(
                    val_fruits
                )
            ].copy()

            score_table = (
                within_fruit_temporal_band_scores(
                    train_df,
                    wl_cols
                )
            )

            selected = select_spaced_bands(
                score_table,
                k=k,
                min_spacing_nm=MIN_BAND_SPACING_NM
            )

            selected_cols = selected[
                "column"
            ].tolist()

            X_train = preprocess_sparse(
                train_df[
                    selected_cols
                ].to_numpy(
                    dtype=float
                ),
                preprocessing
            )

            X_val = preprocess_sparse(
                val_df[
                    selected_cols
                ].to_numpy(
                    dtype=float
                ),
                preprocessing
            )

            pred = fit_predict_ridge(
                X_train,
                train_df[
                    "storage_days_model"
                ].to_numpy(
                    dtype=float
                ),
                X_val,
                alpha
            )

            pred_parts.append(
                pd.DataFrame({
                    "fruit_id":
                        val_df[
                            "fruit_id"
                        ].to_numpy(
                            dtype=int
                        ),

                    "y_true":
                        val_df[
                            "storage_days_model"
                        ].to_numpy(
                            dtype=float
                        ),

                    "y_pred":
                        pred,
                })
            )

        inner_pred = pd.concat(
            pred_parts,
            ignore_index=True
        )

        fb = fruit_balanced_mae(
            inner_pred
        )

        db = day_balanced_mae(
            inner_pred
        )

        mae = float(
            mean_absolute_error(
                inner_pred[
                    "y_true"
                ],
                inner_pred[
                    "y_pred"
                ]
            )
        )

        rows.append({
            "k":
                int(
                    k
                ),

            "preprocessing":
                preprocessing,

            "alpha":
                float(
                    alpha
                ),

            "inner_fruit_balanced_mae":
                fb,

            "inner_day_balanced_mae":
                db,

            "inner_mae":
                mae,
        })

        rank = (
            fb,
            db,
            mae,
            preprocessing,
            float(
                alpha
            ),
        )

        if (
            best is None
            or
            rank
            <
            best[
                "rank"
            ]
        ):

            best = {
                "rank":
                    rank,

                "preprocessing":
                    preprocessing,

                "alpha":
                    float(
                        alpha
                    ),

                "inner_fruit_balanced_mae":
                    fb,

                "inner_day_balanced_mae":
                    db,

                "inner_mae":
                    mae,
            }

    return (
        best,
        pd.DataFrame(
            rows
        )
    )


# 8. OOF estándar: 3, 5, 8, 10, 15 y 20 bandas


In [ ]:
standard_pred_parts = []
standard_selection_rows = []
standard_search_parts = []

all_fruits = set(
    outer[
        "fruit_id"
    ].astype(int)
)

for outer_fold in range(
    OUTER_FOLDS
):

    test_fruits = set(
        outer.loc[
            outer[
                "outer_fold"
            ]
            ==
            outer_fold,
            "fruit_id"
        ].astype(int)
    )

    development_df = data[
        ~data[
            "fruit_id"
        ].isin(
            test_fruits
        )
    ].copy()

    test_df = data[
        data[
            "fruit_id"
        ].isin(
            test_fruits
        )
    ].copy()

    inner_assignment = inner[
        inner[
            "outer_fold"
        ]
        ==
        outer_fold
    ][
        [
            "fruit_id",
            "inner_fold"
        ]
    ].copy()

    assert (
        development_df[
            "fruit_id"
        ].nunique()
        ==
        32
    )

    assert (
        test_df[
            "fruit_id"
        ].nunique()
        ==
        8
    )

    for k in K_VALUES:

        print(
            f"Outer {outer_fold} | k={k}"
        )

        best, search_df = (
            inner_select_standard_sparse(
                development_df,
                inner_assignment,
                k
            )
        )

        # Selección final de bandas usando solamente
        # los 32 frutos de desarrollo.
        score_table = (
            within_fruit_temporal_band_scores(
                development_df,
                wl_cols
            )
        )

        selected = select_spaced_bands(
            score_table,
            k=k,
            min_spacing_nm=MIN_BAND_SPACING_NM
        )

        selected_cols = selected[
            "column"
        ].tolist()

        X_train = preprocess_sparse(
            development_df[
                selected_cols
            ].to_numpy(
                dtype=float
            ),
            best[
                "preprocessing"
            ]
        )

        X_test = preprocess_sparse(
            test_df[
                selected_cols
            ].to_numpy(
                dtype=float
            ),
            best[
                "preprocessing"
            ]
        )

        pred = fit_predict_ridge(
            X_train,
            development_df[
                "storage_days_model"
            ].to_numpy(
                dtype=float
            ),
            X_test,
            best[
                "alpha"
            ]
        )

        standard_pred_parts.append(
            pd.DataFrame({
                "camera_type":
                    CAMERA,

                "protocol":
                    "standard_fruit_disjoint",

                "outer_fold":
                    int(
                        outer_fold
                    ),

                "k":
                    int(
                        k
                    ),

                "model":
                    f"SparseRidge_k{k}",

                "preprocessing":
                    best[
                        "preprocessing"
                    ],

                "alpha":
                    float(
                        best[
                            "alpha"
                        ]
                    ),

                "fruit_id":
                    test_df[
                        "fruit_id"
                    ].to_numpy(
                        dtype=int
                    ),

                "storage_days":
                    test_df[
                        "storage_days_model"
                    ].to_numpy(
                        dtype=int
                    ),

                "y_true":
                    test_df[
                        "storage_days_model"
                    ].to_numpy(
                        dtype=float
                    ),

                "y_pred":
                    pred,
            })
        )

        for _, row in selected.iterrows():

            standard_selection_rows.append({
                "outer_fold":
                    int(
                        outer_fold
                    ),

                "k":
                    int(
                        k
                    ),

                "selection_rank":
                    int(
                        row[
                            "selection_rank"
                        ]
                    ),

                "wavelength_nm":
                    float(
                        row[
                            "wavelength_nm"
                        ]
                    ),

                "column":
                    row[
                        "column"
                    ],

                "selection_score":
                    float(
                        row[
                            "selection_score"
                        ]
                    ),

                "mean_within_fruit_r":
                    float(
                        row[
                            "mean_within_fruit_r"
                        ]
                    ),

                "sign_consistency":
                    float(
                        row[
                            "sign_consistency"
                        ]
                    ),

                "preprocessing_selected":
                    best[
                        "preprocessing"
                    ],

                "alpha_selected":
                    float(
                        best[
                            "alpha"
                        ]
                    ),
            })

        search_df[
            "outer_fold"
        ] = int(
            outer_fold
        )

        standard_search_parts.append(
            search_df
        )

standard_oof = pd.concat(
    standard_pred_parts,
    ignore_index=True
)

standard_selected_bands = pd.DataFrame(
    standard_selection_rows
)

standard_inner_search = pd.concat(
    standard_search_parts,
    ignore_index=True
)

for k in K_VALUES:

    g = standard_oof[
        standard_oof[
            "k"
        ]
        ==
        k
    ]

    assert len(g) == 199
    assert g["fruit_id"].nunique() == 40

print("✅ OOF estándar sparse completo.")


# 9. Métricas OOF estándar


In [ ]:
standard_metric_rows = []

for k, g in standard_oof.groupby(
    "k"
):

    m = metric_bundle(
        g
    )

    m.update({
        "camera_type":
            CAMERA,

        "protocol":
            "standard_fruit_disjoint",

        "model":
            f"SparseRidge_k{int(k)}",

        "k":
            int(
                k
            ),
    })

    standard_metric_rows.append(
        m
    )

standard_metrics = pd.DataFrame(
    standard_metric_rows
).sort_values(
    "k"
)

display(
    standard_metrics
)

standard_metrics.to_csv(
    TABLES /
    "02_sparse_standard_metrics.csv",
    index=False
)

standard_selected_bands.to_csv(
    TABLES /
    "03_sparse_selected_bands_by_outer_fold.csv",
    index=False
)

standard_inner_search.to_csv(
    TABLES /
    "04_sparse_standard_inner_search.csv",
    index=False
)


# 10. Referencia full-spectrum estándar

Se reutilizan exclusivamente las predicciones OOF de `VIS_COR + Ridge` del notebook 02.


In [ ]:
full_standard = baseline_oof[
    (
        baseline_oof[
            "camera_type"
        ]
        ==
        CAMERA
    )
    &
    (
        baseline_oof[
            "model"
        ]
        ==
        "Ridge"
    )
].copy()

assert len(
    full_standard
) == 199

assert full_standard[
    "fruit_id"
].nunique() == 40

full_standard[
    "model"
] = "Full249_Ridge_standard"

full_standard_metrics = metric_bundle(
    full_standard
)

display(
    pd.DataFrame(
        [
            full_standard_metrics
        ]
    )
)


# 11. Null estándar

Se reutiliza el `DummyMedian` fruit-disjoint del notebook 02.


In [ ]:
dummy_standard = baseline_oof[
    (
        baseline_oof[
            "camera_type"
        ]
        ==
        CAMERA
    )
    &
    (
        baseline_oof[
            "model"
        ]
        ==
        "DummyMedian"
    )
].copy()

assert len(
    dummy_standard
) == 199

dummy_standard[
    "model"
] = "DummyMedian_standard"

dummy_standard_metrics = metric_bundle(
    dummy_standard
)


# 12. Session-excluded null correcto

Para cada outer fold y cada día objetivo \(d\):

1. se excluyen los 8 frutos test;
2. del desarrollo se eliminan todas las observaciones con día \(d\);
3. se calcula la media de `storage_days` restante;
4. esa constante predice el test del día \(d\).

Este es el null principal para interpretar el stress test.


In [ ]:
session_null_parts = []

for outer_fold in range(
    OUTER_FOLDS
):

    test_fruits = set(
        outer.loc[
            outer[
                "outer_fold"
            ]
            ==
            outer_fold,
            "fruit_id"
        ].astype(int)
    )

    development_df = data[
        ~data[
            "fruit_id"
        ].isin(
            test_fruits
        )
    ].copy()

    test_df = data[
        data[
            "fruit_id"
        ].isin(
            test_fruits
        )
    ].copy()

    for target_day in sorted(
        test_df[
            "storage_days_model"
        ].unique()
    ):

        train_excluded = development_df[
            development_df[
                "storage_days_model"
            ]
            !=
            target_day
        ]

        test_day = test_df[
            test_df[
                "storage_days_model"
            ]
            ==
            target_day
        ]

        null_value = float(
            train_excluded[
                "storage_days_model"
            ].mean()
        )

        session_null_parts.append(
            pd.DataFrame({
                "camera_type":
                    CAMERA,

                "protocol":
                    "session_excluded",

                "outer_fold":
                    int(
                        outer_fold
                    ),

                "model":
                    "SessionExcluded_DummyMean",

                "fruit_id":
                    test_day[
                        "fruit_id"
                    ].to_numpy(
                        dtype=int
                    ),

                "storage_days":
                    test_day[
                        "storage_days_model"
                    ].to_numpy(
                        dtype=int
                    ),

                "y_true":
                    test_day[
                        "storage_days_model"
                    ].to_numpy(
                        dtype=float
                    ),

                "y_pred":
                    np.repeat(
                        null_value,
                        len(
                            test_day
                        )
                    ),
            })
        )

session_null_oof = pd.concat(
    session_null_parts,
    ignore_index=True
)

assert len(
    session_null_oof
) == 199

session_null_metrics = metric_bundle(
    session_null_oof
)

display(
    pd.DataFrame(
        [
            session_null_metrics
        ]
    )
)


# 13. Full-spectrum session-excluded de referencia

Se usa el Ridge session-excluded del notebook 03.

Este full-spectrum no tiene etapa de selección de bandas.


In [ ]:
full_session = session_oof_full[
    session_oof_full[
        "camera_type"
    ]
    ==
    CAMERA
].copy()

assert len(
    full_session
) == 199

assert full_session[
    "fruit_id"
].nunique() == 40

full_session[
    "model"
] = "Full249_Ridge_session_excluded"

full_session_metrics = metric_bundle(
    full_session
)

display(
    pd.DataFrame(
        [
            full_session_metrics
        ]
    )
)


# 14. Inner tuning session-excluded usando un panel fijo

El panel de cada `outer_fold × k` ya fue seleccionado con los 32 frutos de desarrollo.

Ahora solo se ajustan:

- RAW/SNV
- alpha

con una CV interna que excluye el día objetivo del training.

El panel de bandas **no cambia entre días objetivo**.


In [ ]:
def inner_select_session_excluded_fixed_panel(
    development_df,
    inner_assignment,
    selected_cols
):

    rows = []

    best = None

    for preprocessing, alpha in itertools.product(
        PREPROCESSING,
        RIDGE_ALPHAS
    ):

        pred_parts = []

        for inner_fold in range(
            INNER_FOLDS
        ):

            val_fruits = set(
                inner_assignment.loc[
                    inner_assignment[
                        "inner_fold"
                    ]
                    ==
                    inner_fold,
                    "fruit_id"
                ].astype(int)
            )

            inner_train = development_df[
                ~development_df[
                    "fruit_id"
                ].isin(
                    val_fruits
                )
            ].copy()

            inner_val = development_df[
                development_df[
                    "fruit_id"
                ].isin(
                    val_fruits
                )
            ].copy()

            for target_day in sorted(
                inner_val[
                    "storage_days_model"
                ].unique()
            ):

                train_excluded = inner_train[
                    inner_train[
                        "storage_days_model"
                    ]
                    !=
                    target_day
                ].copy()

                val_day = inner_val[
                    inner_val[
                        "storage_days_model"
                    ]
                    ==
                    target_day
                ].copy()

                X_train = preprocess_sparse(
                    train_excluded[
                        selected_cols
                    ].to_numpy(
                        dtype=float
                    ),
                    preprocessing
                )

                X_val = preprocess_sparse(
                    val_day[
                        selected_cols
                    ].to_numpy(
                        dtype=float
                    ),
                    preprocessing
                )

                pred = fit_predict_ridge(
                    X_train,
                    train_excluded[
                        "storage_days_model"
                    ].to_numpy(
                        dtype=float
                    ),
                    X_val,
                    alpha
                )

                pred_parts.append(
                    pd.DataFrame({
                        "fruit_id":
                            val_day[
                                "fruit_id"
                            ].to_numpy(
                                dtype=int
                            ),

                        "y_true":
                            val_day[
                                "storage_days_model"
                            ].to_numpy(
                                dtype=float
                            ),

                        "y_pred":
                            pred,
                    })
                )

        inner_pred = pd.concat(
            pred_parts,
            ignore_index=True
        )

        fb = fruit_balanced_mae(
            inner_pred
        )

        db = day_balanced_mae(
            inner_pred
        )

        mae = float(
            mean_absolute_error(
                inner_pred[
                    "y_true"
                ],
                inner_pred[
                    "y_pred"
                ]
            )
        )

        rows.append({
            "preprocessing":
                preprocessing,

            "alpha":
                float(
                    alpha
                ),

            "inner_fruit_balanced_mae":
                fb,

            "inner_day_balanced_mae":
                db,

            "inner_mae":
                mae,
        })

        rank = (
            fb,
            db,
            mae,
            preprocessing,
            float(
                alpha
            ),
        )

        if (
            best is None
            or
            rank
            <
            best[
                "rank"
            ]
        ):

            best = {
                "rank":
                    rank,

                "preprocessing":
                    preprocessing,

                "alpha":
                    float(
                        alpha
                    ),

                "inner_fruit_balanced_mae":
                    fb,

                "inner_day_balanced_mae":
                    db,

                "inner_mae":
                    mae,
            }

    return (
        best,
        pd.DataFrame(
            rows
        )
    )


# 15. OOF sparse session-excluded

El panel permanece fijo por outer fold.  
Para cada día objetivo, el ajuste del modelo excluye ese día de los 32 frutos de desarrollo.


In [ ]:
session_sparse_parts = []
session_sparse_config_rows = []
session_sparse_search_parts = []

for outer_fold in range(
    OUTER_FOLDS
):

    test_fruits = set(
        outer.loc[
            outer[
                "outer_fold"
            ]
            ==
            outer_fold,
            "fruit_id"
        ].astype(int)
    )

    development_df = data[
        ~data[
            "fruit_id"
        ].isin(
            test_fruits
        )
    ].copy()

    test_df = data[
        data[
            "fruit_id"
        ].isin(
            test_fruits
        )
    ].copy()

    inner_assignment = inner[
        inner[
            "outer_fold"
        ]
        ==
        outer_fold
    ][
        [
            "fruit_id",
            "inner_fold"
        ]
    ].copy()

    for k in K_VALUES:

        selected_rows = standard_selected_bands[
            (
                standard_selected_bands[
                    "outer_fold"
                ]
                ==
                outer_fold
            )
            &
            (
                standard_selected_bands[
                    "k"
                ]
                ==
                k
            )
        ].sort_values(
            "selection_rank"
        )

        assert len(
            selected_rows
        ) == k

        selected_cols = selected_rows[
            "column"
        ].tolist()

        best, search_df = (
            inner_select_session_excluded_fixed_panel(
                development_df,
                inner_assignment,
                selected_cols
            )
        )

        for target_day in sorted(
            test_df[
                "storage_days_model"
            ].unique()
        ):

            train_excluded = development_df[
                development_df[
                    "storage_days_model"
                ]
                !=
                target_day
            ].copy()

            test_day = test_df[
                test_df[
                    "storage_days_model"
                ]
                ==
                target_day
            ].copy()

            X_train = preprocess_sparse(
                train_excluded[
                    selected_cols
                ].to_numpy(
                    dtype=float
                ),
                best[
                    "preprocessing"
                ]
            )

            X_test = preprocess_sparse(
                test_day[
                    selected_cols
                ].to_numpy(
                    dtype=float
                ),
                best[
                    "preprocessing"
                ]
            )

            pred = fit_predict_ridge(
                X_train,
                train_excluded[
                    "storage_days_model"
                ].to_numpy(
                    dtype=float
                ),
                X_test,
                best[
                    "alpha"
                ]
            )

            session_sparse_parts.append(
                pd.DataFrame({
                    "camera_type":
                        CAMERA,

                    "protocol":
                        "session_excluded_fixed_sparse_panel",

                    "outer_fold":
                        int(
                            outer_fold
                        ),

                    "k":
                        int(
                            k
                        ),

                    "model":
                        f"SparseRidge_k{k}_session_excluded",

                    "preprocessing":
                        best[
                            "preprocessing"
                        ],

                    "alpha":
                        float(
                            best[
                                "alpha"
                            ]
                        ),

                    "fruit_id":
                        test_day[
                            "fruit_id"
                        ].to_numpy(
                            dtype=int
                        ),

                    "storage_days":
                        test_day[
                            "storage_days_model"
                        ].to_numpy(
                            dtype=int
                        ),

                    "y_true":
                        test_day[
                            "storage_days_model"
                        ].to_numpy(
                            dtype=float
                        ),

                    "y_pred":
                        pred,
                })
            )

        session_sparse_config_rows.append({
            "outer_fold":
                int(
                    outer_fold
                ),

            "k":
                int(
                    k
                ),

            "preprocessing":
                best[
                    "preprocessing"
                ],

            "alpha":
                float(
                    best[
                        "alpha"
                    ]
                ),

            "inner_fruit_balanced_mae":
                float(
                    best[
                        "inner_fruit_balanced_mae"
                    ]
                ),
        })

        search_df[
            "outer_fold"
        ] = int(
            outer_fold
        )

        search_df[
            "k"
        ] = int(
            k
        )

        session_sparse_search_parts.append(
            search_df
        )

session_sparse_oof = pd.concat(
    session_sparse_parts,
    ignore_index=True
)

session_sparse_configs = pd.DataFrame(
    session_sparse_config_rows
)

session_sparse_search = pd.concat(
    session_sparse_search_parts,
    ignore_index=True
)

for k in K_VALUES:

    g = session_sparse_oof[
        session_sparse_oof[
            "k"
        ]
        ==
        k
    ]

    assert len(g) == 199
    assert g["fruit_id"].nunique() == 40

print("✅ Sparse session-excluded completo.")


# 16. Métricas session-excluded sparse


In [ ]:
session_sparse_metric_rows = []

for k, g in session_sparse_oof.groupby(
    "k"
):

    m = metric_bundle(
        g
    )

    m.update({
        "camera_type":
            CAMERA,

        "protocol":
            "session_excluded_fixed_sparse_panel",

        "model":
            f"SparseRidge_k{int(k)}_session_excluded",

        "k":
            int(
                k
            ),
    })

    session_sparse_metric_rows.append(
        m
    )

session_sparse_metrics = pd.DataFrame(
    session_sparse_metric_rows
).sort_values(
    "k"
)

display(
    session_sparse_metrics
)

session_sparse_metrics.to_csv(
    TABLES /
    "05_sparse_session_excluded_metrics.csv",
    index=False
)

session_sparse_configs.to_csv(
    TABLES /
    "06_sparse_session_excluded_configs.csv",
    index=False
)

session_sparse_search.to_csv(
    TABLES /
    "07_sparse_session_excluded_inner_search.csv",
    index=False
)


# 17. Tabla maestra de comparación


In [ ]:
master_rows = []

row = full_standard_metrics.copy()
row.update({
    "protocol":
        "standard_fruit_disjoint",

    "model":
        "Full249_Ridge_standard",

    "k":
        249,
})
master_rows.append(
    row
)

row = dummy_standard_metrics.copy()
row.update({
    "protocol":
        "standard_fruit_disjoint",

    "model":
        "DummyMedian_standard",

    "k":
        0,
})
master_rows.append(
    row
)

for _, r in standard_metrics.iterrows():

    master_rows.append(
        r.to_dict()
    )

row = full_session_metrics.copy()
row.update({
    "protocol":
        "session_excluded",

    "model":
        "Full249_Ridge_session_excluded",

    "k":
        249,
})
master_rows.append(
    row
)

row = session_null_metrics.copy()
row.update({
    "protocol":
        "session_excluded",

    "model":
        "SessionExcluded_DummyMean",

    "k":
        0,
})
master_rows.append(
    row
)

for _, r in session_sparse_metrics.iterrows():

    master_rows.append(
        r.to_dict()
    )

comparison_metrics = pd.DataFrame(
    master_rows
)

display(
    comparison_metrics.sort_values(
        [
            "protocol",
            "k"
        ]
    )
)

comparison_metrics.to_csv(
    TABLES /
    "08_full_sparse_null_comparison_metrics.csv",
    index=False
)


# 18. Bootstrap pareado por fruto

Para cada panel sparse se compara:

### Protocolo estándar
\[
\Delta = MAE_{sparse} - MAE_{full249}
\]

### Session-excluded
\[
\Delta = MAE_{sparse} - MAE_{full249}
\]

y adicionalmente:

\[
\Delta_{null} = MAE_{sparse} - MAE_{SessionExcludedNull}
\]

Valores negativos contra el null significan que el sensor sparse supera al null sometido a la misma exclusión.


In [ ]:
def per_fruit_mae_series(
    df
):

    z = df.copy()

    z[
        "abs_error"
    ] = np.abs(
        z[
            "y_true"
        ]
        -
        z[
            "y_pred"
        ]
    )

    return (
        z.groupby(
            "fruit_id"
        )[
            "abs_error"
        ]
        .mean()
    )


def paired_bootstrap(
    a,
    b,
    label,
    n_boot=5000,
    seed=20260818
):

    common = sorted(
        set(
            a.index
        )
        &
        set(
            b.index
        )
    )

    av = a.loc[
        common
    ].to_numpy(
        dtype=float
    )

    bv = b.loc[
        common
    ].to_numpy(
        dtype=float
    )

    diff = (
        av
        -
        bv
    )

    rng = np.random.default_rng(
        seed
    )

    boot = np.empty(
        n_boot,
        dtype=float
    )

    for i in range(
        n_boot
    ):

        idx = rng.integers(
            0,
            len(
                diff
            ),
            size=len(
                diff
            )
        )

        boot[
            i
        ] = np.mean(
            diff[
                idx
            ]
        )

    return {
        "comparison":
            label,

        "n_fruits":
            int(
                len(
                    diff
                )
            ),

        "mean_delta_mae_days":
            float(
                np.mean(
                    diff
                )
            ),

        "median_delta_mae_days":
            float(
                np.median(
                    diff
                )
            ),

        "ci95_low":
            float(
                np.percentile(
                    boot,
                    2.5
                )
            ),

        "ci95_high":
            float(
                np.percentile(
                    boot,
                    97.5
                )
            ),

        "fraction_first_better":
            float(
                np.mean(
                    diff
                    <
                    0
                )
            ),
    }


bootstrap_rows = []

full_std_pf = per_fruit_mae_series(
    full_standard
)

full_sess_pf = per_fruit_mae_series(
    full_session
)

null_sess_pf = per_fruit_mae_series(
    session_null_oof
)

for idx, k in enumerate(
    K_VALUES
):

    sparse_std = standard_oof[
        standard_oof[
            "k"
        ]
        ==
        k
    ]

    sparse_sess = session_sparse_oof[
        session_sparse_oof[
            "k"
        ]
        ==
        k
    ]

    sparse_std_pf = per_fruit_mae_series(
        sparse_std
    )

    sparse_sess_pf = per_fruit_mae_series(
        sparse_sess
    )

    bootstrap_rows.append(
        paired_bootstrap(
            sparse_std_pf,
            full_std_pf,
            (
                f"standard sparse k={k} "
                "minus full249"
            ),
            n_boot=BOOTSTRAP_REPS,
            seed=RANDOM_SEED + idx
        )
    )

    bootstrap_rows[-1][
        "protocol"
    ] = "standard"

    bootstrap_rows[-1][
        "k"
    ] = int(
        k
    )

    bootstrap_rows.append(
        paired_bootstrap(
            sparse_sess_pf,
            full_sess_pf,
            (
                f"session-excluded sparse k={k} "
                "minus full249"
            ),
            n_boot=BOOTSTRAP_REPS,
            seed=RANDOM_SEED + 100 + idx
        )
    )

    bootstrap_rows[-1][
        "protocol"
    ] = "session_excluded_vs_full"

    bootstrap_rows[-1][
        "k"
    ] = int(
        k
    )

    bootstrap_rows.append(
        paired_bootstrap(
            sparse_sess_pf,
            null_sess_pf,
            (
                f"session-excluded sparse k={k} "
                "minus session null"
            ),
            n_boot=BOOTSTRAP_REPS,
            seed=RANDOM_SEED + 200 + idx
        )
    )

    bootstrap_rows[-1][
        "protocol"
    ] = "session_excluded_vs_null"

    bootstrap_rows[-1][
        "k"
    ] = int(
        k
    )

bootstrap_results = pd.DataFrame(
    bootstrap_rows
)

display(
    bootstrap_results
)

bootstrap_results.to_csv(
    TABLES /
    "09_paired_fruit_bootstrap_sparse_comparisons.csv",
    index=False
)


# 19. Retención de desempeño vs 249 bandas

Se reporta:

\[
\Delta MAE = MAE_{sparse} - MAE_{full}
\]

y:

\[
ratio = 
rac{MAE_{sparse}}{MAE_{full}}
\]

No se declara equivalencia estadística automática.


In [ ]:
retention_rows = []

full_std_mae = float(
    full_standard_metrics[
        "fruit_balanced_mae"
    ]
)

full_sess_mae = float(
    full_session_metrics[
        "fruit_balanced_mae"
    ]
)

for _, row in standard_metrics.iterrows():

    retention_rows.append({
        "protocol":
            "standard",

        "k":
            int(
                row[
                    "k"
                ]
            ),

        "sparse_fruit_balanced_mae":
            float(
                row[
                    "fruit_balanced_mae"
                ]
            ),

        "full249_fruit_balanced_mae":
            full_std_mae,

        "delta_mae_days":
            float(
                row[
                    "fruit_balanced_mae"
                ]
                -
                full_std_mae
            ),

        "mae_ratio_sparse_over_full":
            float(
                row[
                    "fruit_balanced_mae"
                ]
                /
                full_std_mae
            ),
    })

for _, row in session_sparse_metrics.iterrows():

    retention_rows.append({
        "protocol":
            "session_excluded",

        "k":
            int(
                row[
                    "k"
                ]
            ),

        "sparse_fruit_balanced_mae":
            float(
                row[
                    "fruit_balanced_mae"
                ]
            ),

        "full249_fruit_balanced_mae":
            full_sess_mae,

        "delta_mae_days":
            float(
                row[
                    "fruit_balanced_mae"
                ]
                -
                full_sess_mae
            ),

        "mae_ratio_sparse_over_full":
            float(
                row[
                    "fruit_balanced_mae"
                ]
                /
                full_sess_mae
            ),
    })

retention = pd.DataFrame(
    retention_rows
)

display(
    retention
)

retention.to_csv(
    TABLES /
    "10_sparse_performance_retention.csv",
    index=False
)


# 20. Figura — MAE vs número de bandas


In [ ]:
fig, ax = plt.subplots(
    figsize=(8.5, 5)
)

std = standard_metrics.sort_values(
    "k"
)

ses = session_sparse_metrics.sort_values(
    "k"
)

ax.plot(
    std[
        "k"
    ],
    std[
        "fruit_balanced_mae"
    ],
    marker="o",
    label="Sparse estándar"
)

ax.plot(
    ses[
        "k"
    ],
    ses[
        "fruit_balanced_mae"
    ],
    marker="s",
    label="Sparse session-excluded"
)

ax.axhline(
    full_std_mae,
    linestyle="--",
    label="Full249 estándar"
)

ax.axhline(
    full_sess_mae,
    linestyle=":",
    label="Full249 session-excluded"
)

ax.axhline(
    float(
        session_null_metrics[
            "fruit_balanced_mae"
        ]
    ),
    linestyle="-.",
    label="Null session-excluded"
)

ax.set_xlabel(
    "Número de bandas"
)

ax.set_ylabel(
    "MAE fruit-balanced (días)"
)

ax.set_title(
    "VIS_COR — desempeño vs número de longitudes de onda"
)

ax.legend(
    fontsize=8
)

ax.grid(
    alpha=0.2
)

fig.tight_layout()

fig.savefig(
    FIGURES /
    "02_mae_vs_number_of_bands.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()


# 21. Estabilidad de bandas entre outer folds


In [ ]:
selection_frequency = (
    standard_selected_bands.groupby(
        [
            "k",
            "wavelength_nm"
        ]
    )
    .agg(
        n_outer_folds_selected=(
            "outer_fold",
            "nunique"
        ),
        mean_selection_rank=(
            "selection_rank",
            "mean"
        ),
        mean_selection_score=(
            "selection_score",
            "mean"
        ),
        mean_sign_consistency=(
            "sign_consistency",
            "mean"
        ),
    )
    .reset_index()
    .sort_values(
        [
            "k",
            "n_outer_folds_selected",
            "mean_selection_rank"
        ],
        ascending=[
            True,
            False,
            True
        ]
    )
)

display(
    selection_frequency
)

selection_frequency.to_csv(
    TABLES /
    "11_band_selection_stability.csv",
    index=False
)


## 22. Figura — bandas seleccionadas por outer fold


In [ ]:
for k in K_VALUES:

    g = standard_selected_bands[
        standard_selected_bands[
            "k"
        ]
        ==
        k
    ]

    fig, ax = plt.subplots(
        figsize=(10, 4.5)
    )

    for fold in range(
        OUTER_FOLDS
    ):

        f = g[
            g[
                "outer_fold"
            ]
            ==
            fold
        ]

        ax.scatter(
            f[
                "wavelength_nm"
            ],
            np.repeat(
                fold,
                len(
                    f
                )
            ),
            s=45
        )

    ax.set_xlabel(
        "Longitud de onda seleccionada (nm)"
    )

    ax.set_ylabel(
        "Outer fold"
    )

    ax.set_yticks(
        range(
            OUTER_FOLDS
        )
    )

    ax.set_title(
        f"VIS_COR — estabilidad de selección, k={k}"
    )

    ax.grid(
        axis="x",
        alpha=0.15
    )

    fig.tight_layout()

    fig.savefig(
        FIGURES /
        f"03_selected_wavelengths_outer_folds_k{k}.png",
        dpi=300,
        bbox_inches="tight"
    )

    plt.show()


# 23. Error por día para cada tamaño de sensor


In [ ]:
day_rows = []

for protocol_name, pred_df in [
    (
        "standard",
        standard_oof
    ),
    (
        "session_excluded",
        session_sparse_oof
    ),
]:

    for (
        k,
        day
    ), g in pred_df.groupby(
        [
            "k",
            "storage_days"
        ]
    ):

        err = (
            g[
                "y_pred"
            ].to_numpy(
                dtype=float
            )
            -
            g[
                "y_true"
            ].to_numpy(
                dtype=float
            )
        )

        day_rows.append({
            "protocol":
                protocol_name,

            "k":
                int(
                    k
                ),

            "storage_days":
                int(
                    day
                ),

            "n":
                int(
                    len(
                        g
                    )
                ),

            "mae":
                float(
                    np.mean(
                        np.abs(
                            err
                        )
                    )
                ),

            "bias":
                float(
                    np.mean(
                        err
                    )
                ),
        })

day_metrics = pd.DataFrame(
    day_rows
)

display(
    day_metrics
)

day_metrics.to_csv(
    TABLES /
    "12_sparse_error_bias_by_day.csv",
    index=False
)


# 24. Selección exploratoria de un panel compacto

Se genera una tabla Pareto.

No se fuerza automáticamente una conclusión de equivalencia.

Como criterio de ingeniería **descriptivo**, se marca si el panel queda a no más de `0.10 días` del MAE full-spectrum estándar.

Ese margen no es una prueba estadística.


In [ ]:
ENGINEERING_DELTA_DAYS = 0.10

pareto = standard_metrics[
    [
        "k",
        "fruit_balanced_mae",
        "mae",
        "rmse",
        "r2",
        "within_1_day_accuracy",
    ]
].copy()

pareto[
    "delta_vs_full249_days"
] = (
    pareto[
        "fruit_balanced_mae"
    ]
    -
    full_std_mae
)

pareto[
    "within_0p10_day_engineering_screen"
] = (
    pareto[
        "delta_vs_full249_days"
    ]
    <=
    ENGINEERING_DELTA_DAYS
)

display(
    pareto
)

pareto.to_csv(
    TABLES /
    "13_sparse_sensor_pareto_table.csv",
    index=False
)

eligible = pareto[
    pareto[
        "within_0p10_day_engineering_screen"
    ]
]

if len(
    eligible
) > 0:

    exploratory_k = int(
        eligible[
            "k"
        ].min()
    )

    print(
        "Panel compacto exploratorio:",
        exploratory_k,
        "bandas"
    )

else:

    exploratory_k = None

    print(
        "Ningún panel sparse quedó dentro de +0.10 días "
        "del full-spectrum."
    )


# 25. Panel consenso descriptivo con los 40 frutos

Después de terminar la estimación OOF, se calcula un panel consenso utilizando todos los frutos.

Este panel sirve para proponer un **sensor candidato futuro**, pero su desempeño no se considera validación independiente porque usa todo el dataset para el diseño.


In [ ]:
consensus_rows = []

all_scores = (
    within_fruit_temporal_band_scores(
        data,
        wl_cols
    )
)

for k in K_VALUES:

    selected = select_spaced_bands(
        all_scores,
        k=k,
        min_spacing_nm=MIN_BAND_SPACING_NM
    )

    for _, row in selected.iterrows():

        consensus_rows.append({
            "k":
                int(
                    k
                ),

            "selection_rank":
                int(
                    row[
                        "selection_rank"
                    ]
                ),

            "wavelength_nm":
                float(
                    row[
                        "wavelength_nm"
                    ]
                ),

            "selection_score":
                float(
                    row[
                        "selection_score"
                    ]
                ),

            "mean_within_fruit_r":
                float(
                    row[
                        "mean_within_fruit_r"
                    ]
                ),

            "sign_consistency":
                float(
                    row[
                        "sign_consistency"
                    ]
                ),
        })

consensus_panels = pd.DataFrame(
    consensus_rows
)

display(
    consensus_panels
)

consensus_panels.to_csv(
    TABLES /
    "14_consensus_candidate_wavelength_panels.csv",
    index=False
)

if exploratory_k is not None:

    print(
        "\nPanel exploratorio k=",
        exploratory_k,
        sep=""
    )

    display(
        consensus_panels[
            consensus_panels[
                "k"
            ]
            ==
            exploratory_k
        ]
    )


# 26. Comparación del panel sparse contra el null session-excluded

Esta tabla es importante porque evita interpretar el stress test contra un null que no sufrió la misma exclusión.


In [ ]:
null_mae = float(
    session_null_metrics[
        "fruit_balanced_mae"
    ]
)

session_null_comparison = (
    session_sparse_metrics[
        [
            "k",
            "fruit_balanced_mae",
            "r2",
            "within_1_day_accuracy",
        ]
    ]
    .copy()
)

session_null_comparison[
    "session_null_fruit_balanced_mae"
] = null_mae

session_null_comparison[
    "improvement_over_session_null_days"
] = (
    null_mae
    -
    session_null_comparison[
        "fruit_balanced_mae"
    ]
)

session_null_comparison[
    "relative_improvement_over_null"
] = (
    session_null_comparison[
        "improvement_over_session_null_days"
    ]
    /
    null_mae
)

display(
    session_null_comparison
)

session_null_comparison.to_csv(
    TABLES /
    "15_sparse_session_excluded_vs_correct_null.csv",
    index=False
)


# 27. Guardar predicciones centrales


In [ ]:
STANDARD_OOF_PATH = (
    RESULTS /
    "kaki_viscor_sparse_standard_oof.csv"
)

SESSION_SPARSE_OOF_PATH = (
    RESULTS /
    "kaki_viscor_sparse_session_excluded_oof.csv"
)

SESSION_NULL_OOF_PATH = (
    RESULTS /
    "kaki_session_excluded_dummy_mean_oof.csv"
)

COMPARISON_METRICS_PATH = (
    RESULTS /
    "kaki_sparse_sensor_comparison_metrics.csv"
)

standard_oof.to_csv(
    STANDARD_OOF_PATH,
    index=False
)

session_sparse_oof.to_csv(
    SESSION_SPARSE_OOF_PATH,
    index=False
)

session_null_oof.to_csv(
    SESSION_NULL_OOF_PATH,
    index=False
)

comparison_metrics.to_csv(
    COMPARISON_METRICS_PATH,
    index=False
)

print("✅ Resultados centrales guardados.")


# 28. Resumen científico automático


In [ ]:
def json_safe(
    obj
):

    if isinstance(
        obj,
        dict
    ):
        return {
            str(k):
                json_safe(
                    v
                )
            for k, v in obj.items()
        }

    if isinstance(
        obj,
        (
            list,
            tuple
        )
    ):
        return [
            json_safe(
                v
            )
            for v in obj
        ]

    if isinstance(
        obj,
        np.integer
    ):
        return int(
            obj
        )

    if isinstance(
        obj,
        np.floating
    ):
        return (
            None
            if np.isnan(
                obj
            )
            else float(
                obj
            )
        )

    if isinstance(
        obj,
        np.bool_
    ):
        return bool(
            obj
        )

    if isinstance(
        obj,
        np.ndarray
    ):
        return json_safe(
            obj.tolist()
        )

    return obj


summary = {
    "project":
        PROJECT_NAME,

    "notebook":
        NOTEBOOK_ID,

    "run_signature":
        RUN_SIGNATURE,

    "timestamp":
        datetime.now().isoformat(
            timespec="seconds"
        ),

    "camera":
        CAMERA,

    "full_spectrum_bands":
        249,

    "candidate_sparse_band_counts":
        K_VALUES,

    "minimum_band_spacing_nm":
        MIN_BAND_SPACING_NM,

    "band_selection": {
        "type":
            "training-only longitudinal within-fruit temporal correlation",

        "score":
            (
                "absolute mean within-fruit temporal correlation "
                "multiplied by sign consistency"
            ),

        "standard_nested_selection":
            True,

        "sparse_preprocessing_candidates":
            PREPROCESSING,

        "sg1_excluded_reason":
            (
                "Sparse discrete-band sensor simulation should not rely "
                "on dense-neighborhood Savitzky-Golay derivatives."
            ),
    },

    "standard_full249":
        full_standard_metrics,

    "standard_sparse":
        standard_metrics.to_dict(
            orient="records"
        ),

    "session_excluded_full249":
        full_session_metrics,

    "session_excluded_correct_null":
        session_null_metrics,

    "session_excluded_sparse":
        session_sparse_metrics.to_dict(
            orient="records"
        ),

    "retention":
        retention.to_dict(
            orient="records"
        ),

    "paired_bootstrap":
        bootstrap_results.to_dict(
            orient="records"
        ),

    "engineering_screen_delta_days":
        ENGINEERING_DELTA_DAYS,

    "exploratory_smallest_k_within_engineering_screen":
        exploratory_k,

    "limitations": [
        (
            "Sparse-band performance is internal to one experimental campaign."
        ),
        (
            "The fixed sparse panel for the session-excluded stress test "
            "was designed using all days of the outer development fruits; "
            "therefore this stress test does not fully remove session exposure "
            "from the sensor-design stage."
        ),
        (
            "The 0.10-day engineering screen is descriptive and is not a "
            "formal statistical equivalence margin."
        ),
        (
            "Consensus panels derived from all 40 fruits are candidate sensor "
            "designs only and have no independent validation."
        ),
        (
            "External batch/campaign validation is still required before "
            "deployment claims."
        ),
    ],
}

SUMMARY_PATH = (
    RESULTS /
    "kaki_wavelength_selection_sensor_simplification_summary.json"
)

SUMMARY_PATH.write_text(
    json.dumps(
        json_safe(
            summary
        ),
        indent=2,
        ensure_ascii=False
    ),
    encoding="utf-8"
)

print(
    json.dumps(
        json_safe(
            summary
        ),
        indent=2,
        ensure_ascii=False
    )
)


# 29. Notas automáticas de revisión


In [ ]:
review_lines = [
    "KAKI — WAVELENGTH SELECTION / SENSOR SIMPLIFICATION",
    "",
    f"Camera: {CAMERA}",
    f"Full bands: {len(wavelengths)}",
    f"Sparse k values: {K_VALUES}",
    f"Minimum spacing: {MIN_BAND_SPACING_NM} nm",
    "",
    (
        "Full249 standard fruit-balanced MAE = "
        f"{full_std_mae:.4f} d"
    ),
    (
        "Full249 session-excluded fruit-balanced MAE = "
        f"{full_sess_mae:.4f} d"
    ),
    (
        "Correct session-excluded null fruit-balanced MAE = "
        f"{null_mae:.4f} d"
    ),
    "",
    "STANDARD SPARSE:",
]

for _, r in standard_metrics.iterrows():

    review_lines.append(
        (
            f"  k={int(r['k']):2d}: "
            f"FB-MAE={r['fruit_balanced_mae']:.4f}, "
            f"delta-full={r['fruit_balanced_mae']-full_std_mae:+.4f}, "
            f"R2={r['r2']:.4f}"
        )
    )

review_lines.extend([
    "",
    "SESSION-EXCLUDED SPARSE:",
])

for _, r in session_sparse_metrics.iterrows():

    review_lines.append(
        (
            f"  k={int(r['k']):2d}: "
            f"FB-MAE={r['fruit_balanced_mae']:.4f}, "
            f"delta-full={r['fruit_balanced_mae']-full_sess_mae:+.4f}, "
            f"improvement-null={null_mae-r['fruit_balanced_mae']:+.4f}, "
            f"R2={r['r2']:.4f}"
        )
    )

review_lines.extend([
    "",
    (
        "Exploratory smallest panel within +0.10 d of "
        f"full standard: {exploratory_k}"
    ),
    "",
    "Important:",
    (
        "Session-excluded sparse uses a fixed panel selected from outer "
        "development fruits. Band design therefore saw all development days; "
        "only model fitting/tuning excludes the target day."
    ),
    (
        "Do not call the consensus all-fruit panel independently validated."
    ),
    (
        "Do not proceed to CNN before reviewing this package."
    ),
])

REVIEW_PATH = (
    RESULTS /
    "WAVELENGTH_SELECTION_REVIEW_NOTES.txt"
)

REVIEW_PATH.write_text(
    "\n".join(
        review_lines
    )
    +
    "\n",
    encoding="utf-8"
)

print(
    REVIEW_PATH.read_text()
)
